# Work-Life Firewall Kaggle Runner

Run this notebook top-to-bottom in Kaggle with GPU enabled.

## Required Kaggle Secrets
- `WANDB_API_KEY`
- `HF_TOKEN` (optional, only needed for model upload)


In [9]:
import os
import shutil
import subprocess
from pathlib import Path

# Override via env vars if needed.
GITHUB_REPO = os.environ.get('GITHUB_REPO', 'https://github.com/EchoOfCode/meta_hackathon.git')
GITHUB_BRANCH = os.environ.get('GITHUB_BRANCH', 'main')
WORK_REPO_DIR = Path('/kaggle/working/meta_hackathon')
SAFE_CWD = Path('/kaggle/working')


def _run(cmd, cwd=None):
    run_cwd = str(cwd or SAFE_CWD)
    proc = subprocess.run(cmd, cwd=run_cwd, text=True, capture_output=True)
    if proc.returncode != 0:
        raise RuntimeError(
            f"Command failed: {' '.join(cmd)}\nSTDOUT:\n{proc.stdout}\nSTDERR:\n{proc.stderr}"
        )
    return proc


def _repo_url_with_token(repo_url: str) -> str:
    token = os.environ.get('GITHUB_TOKEN')
    if token and repo_url.startswith('https://github.com/'):
        return repo_url.replace('https://', f'https://{token}@', 1)
    return repo_url


def sync_repo_from_github() -> Path:
    """Always get latest branch tip into /kaggle/working/meta_hackathon."""
    repo_url = _repo_url_with_token(GITHUB_REPO)

    # Ensure we are never inside a path that may be deleted during sync.
    SAFE_CWD.mkdir(parents=True, exist_ok=True)
    os.chdir(SAFE_CWD)

    if (WORK_REPO_DIR / '.git').exists():
        _run(['git', 'fetch', 'origin', GITHUB_BRANCH], cwd=WORK_REPO_DIR)
        _run(['git', 'reset', '--hard', f'origin/{GITHUB_BRANCH}'], cwd=WORK_REPO_DIR)
    else:
        if WORK_REPO_DIR.exists():
            shutil.rmtree(WORK_REPO_DIR)
        _run([
            'git', 'clone', '--depth', '1', '--branch', GITHUB_BRANCH, repo_url, str(WORK_REPO_DIR)
        ], cwd=SAFE_CWD)

    return WORK_REPO_DIR


def find_repo_dir_from_inputs() -> Path:
    """Fallback for offline runs using attached Kaggle dataset input."""
    candidates = []

    working = Path('/kaggle/working')
    if working.exists():
        candidates.append(working)
        candidates.extend([p for p in working.iterdir() if p.is_dir()])

    input_root = Path('/kaggle/input')
    if input_root.exists():
        candidates.append(input_root)
        for ds in input_root.iterdir():
            if ds.is_dir():
                candidates.append(ds)
                candidates.extend([p for p in ds.iterdir() if p.is_dir()])

    for c in candidates:
        if (c / 'openenv.yaml').exists() and (c / 'training').exists():
            return c

    raise FileNotFoundError(
        'Could not find project root with openenv.yaml + training/. '
        'Attach dataset input or enable internet for GitHub sync.'
    )


# Default behavior: pull latest from GitHub each run.
USE_GITHUB_SYNC = os.environ.get('USE_GITHUB_SYNC', '1') == '1'

if USE_GITHUB_SYNC:
    try:
        REPO_DIR = sync_repo_from_github()
        print('Synced latest code from GitHub:', GITHUB_REPO, '@', GITHUB_BRANCH)
    except Exception as exc:
        print('GitHub sync failed, falling back to /kaggle/input snapshot:')
        print(exc)
        REPO_DIR = find_repo_dir_from_inputs()
else:
    REPO_DIR = find_repo_dir_from_inputs()

# /kaggle/input is read-only; copy to /kaggle/working for training outputs.
if str(REPO_DIR).startswith('/kaggle/input'):
    if WORK_REPO_DIR.exists() and WORK_REPO_DIR != REPO_DIR:
        shutil.rmtree(WORK_REPO_DIR)
    shutil.copytree(REPO_DIR, WORK_REPO_DIR, dirs_exist_ok=True)
    REPO_DIR = WORK_REPO_DIR

os.chdir(REPO_DIR)
print('CWD:', Path.cwd())
print('Using project root:', REPO_DIR)
!python --version
!nvidia-smi

Synced latest code from GitHub: https://github.com/EchoOfCode/meta_hackathon.git @ main
CWD: /kaggle/working/meta_hackathon
Using project root: /kaggle/working/meta_hackathon
Python 3.12.12
Sun Apr 26 03:45:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8              9W /   70W 

In [10]:
!pip install -r requirements.txt
!pip install -q openenv trl==0.23.1 unsloth bitsandbytes huggingface_hub

In [11]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()

# WandB token (required for tracking)
os.environ['WANDB_API_KEY'] = secrets.get_secret('WANDB_API_KEY')
os.environ['WANDB_PROJECT'] = 'meta_hackathon'

# HF token is optional; needed only if you push model from Kaggle
try:
    os.environ['HF_TOKEN'] = secrets.get_secret('HF_TOKEN')
    os.environ['HUGGINGFACE_HUB_TOKEN'] = os.environ['HF_TOKEN']
    print('HF token loaded from Kaggle Secrets')
except Exception:
    print('HF token not set (this is okay if you are not uploading model now)')

print('WandB key loaded:', bool(os.environ.get('WANDB_API_KEY')))

HF token loaded from Kaggle Secrets
WandB key loaded: True


In [12]:
# Fast stable run for Kaggle T4
!python training/train.py --mode real --speed-preset fast --size-preset medium --steps 300 --run-name meta-kaggle-final --wandb-project meta_hackathon

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
2026-04-26 03:45:53.182429: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777175153.204707     600 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777175153.213304     600 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777175153.233121     600 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777175153.233178     600 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid li

In [13]:
import json
from pathlib import Path

metrics_path = Path('evaluation/results/training_metrics.json')
metrics = json.loads(metrics_path.read_text())
print('Metrics file:', metrics_path)
print('Mode:', metrics.get('mode'))
print('Model:', metrics.get('model_id'))
print('Train runtime (s):', metrics.get('train_runtime_seconds'))
print('WandB URL:', metrics.get('wandb_run_url'))
print('Loss points:', len(metrics.get('loss_curve', [])))
print('Reward points:', len(metrics.get('reward_curve', [])))

Metrics file: evaluation/results/training_metrics.json
Mode: real
Model: Qwen/Qwen2.5-1.5B-Instruct
Train runtime (s): 3432.3418
WandB URL: https://wandb.ai/yusufindian09-aaa/meta_hackathon/runs/f44p90zl
Loss points: 60
Reward points: 60


In [14]:
#!python evaluation/evaluate.py --episodes 50
!python evaluation/plot_results.py
!ls -lah evaluation/results

Saved PNG plots in evaluation/results/
total 288K
drwxr-xr-x 2 root root 4.0K Apr 26 03:45 .
drwxr-xr-x 4 root root 4.0K Apr 26 02:22 ..
-rw-r--r-- 1 root root  34K Apr 26 02:22 component_breakdown_20260426.png
-rw-r--r-- 1 root root  34K Apr 26 04:43 component_breakdown.png
-rw-r--r-- 1 root root  51K Apr 26 04:43 decision_heatmap.png
-rw-r--r-- 1 root root  22K Apr 26 04:43 energy_trajectory.png
-rw-r--r-- 1 root root 6.9K Apr 26 02:22 evaluation_summary.json
-rw-r--r-- 1 root root  14K Apr 26 04:43 loss_curve.png
-rw-r--r-- 1 root root  46K Apr 26 04:43 reward_curve.png
-rw-r--r-- 1 root root  57K Apr 26 04:43 training_metrics.json


In [15]:
# Package submission artifacts for download
!zip -r submission_artifacts.zip README.md openenv.yaml environment training evaluation/results -x "*/__pycache__/*" "*.pyc" ".ipynb_checkpoints/*"
!ls -lah submission_artifacts.zip

updating: README.md (deflated 60%)
updating: openenv.yaml (deflated 29%)
updating: environment/ (stored 0%)
updating: environment/events.py (deflated 74%)
updating: environment/state.py (deflated 66%)
updating: environment/env.py (deflated 64%)
updating: environment/judge.py (deflated 55%)
updating: environment/consequences.py (deflated 65%)
updating: environment/reward.py (deflated 67%)
updating: environment/__init__.py (deflated 23%)
updating: training/ (stored 0%)
updating: training/push_to_hf.py (deflated 50%)
updating: training/__init__.py (stored 0%)
updating: training/rollout.py (deflated 50%)
updating: training/train.py (deflated 70%)
updating: training/train.ipynb (deflated 70%)
updating: training/grpo_config.py (deflated 43%)
updating: evaluation/results/ (stored 0%)
updating: evaluation/results/training_metrics.json (deflated 94%)
updating: evaluation/results/energy_trajectory.png (deflated 12%)
updating: evaluation/results/decision_heatmap.png (deflated 12%)
updating: evalu

In [16]:
# Optional: upload model to HF Model Hub
# Uncomment and edit repo id when needed
!python training/push_to_hf.py --repo-id YUS200619/meta_hackathon-qwen-model --folder checkpoints/final

Processing Files (0 / 0)      : |                  |  0.00B /  0.00B            
New Data Upload               : |                  |  0.00B /  0.00B            

  ...ints/final/tokenizer.json: 100%|██████████████| 11.4MB / 11.4MB            

Processing Files (1 / 1)      :  13%|█▊            | 11.4MB / 85.3MB, 11.4MB/s  


  ...adapter_model.safetensors:   0%|              | 45.7kB / 73.9MB            

  ...ints/final/tokenizer.json: 100%|██████████████| 11.4MB / 11.4MB            


Processing Files (1 / 2)      :  13%|█▉            | 11.5MB / 85.3MB, 9.55MB/s  

  ...ints/final/tokenizer.json: 100%|██████████████| 11.4MB / 11.4MB            


  ...adapter_model.safetensors:   0%|              | 45.7kB / 73.9MB            

  ...ints/final/tokenizer.json: 100%|██████████████| 11.4MB / 11.4MB            


  ...adapter_model.safetensors:   0%|              | 45.7kB / 73.9MB            

  ...ints/final/tokenizer.json: 100%|██████████████| 11.4MB / 11.4MB            


  ...adapter